# 03_feature_engineering — 特徴量エンジニアリング

フロントエンドの学習パイプライン（`python-api/routers/train.py`）と
同一のロジックを使用して特徴量を生成する。

## パイプライン（INV-01）
```
DB読み込み
  → load_ultimate_training_frame()
  → add_derived_features()          # keiba_ai.feature_engineering
  → POST_RACE_FIELDS / FUTURE_FIELDS 除外
  → LightGBMFeatureOptimizer.fit_transform()
  → feature_summary.csv 出力
```

## 特徴量カテゴリ
| カテゴリ | 関数 |
|---|---|
| 馬能力 | `_feh_recent_form`, `_feh_horse_aptitude` |
| 騎手能力 | `_feh_entity_career`, `_feh_entity_recent30` |
| 調教師能力 | `_feh_entity_career` |
| レース条件 | `_fe_id_season`, `_fe_course` |
| 人気・オッズ | `_fe_market` |
| 期待値 | Kelly 計算（06_prediction） |
| 時系列 | expanding window / rolling window |

In [1]:
import sys, json, os
from pathlib import Path
_NB_DIR = Path().resolve()
if _NB_DIR.name != 'notebooks': _NB_DIR = _NB_DIR.parent
if str(_NB_DIR) not in sys.path: sys.path.insert(0, str(_NB_DIR))
from utils.nb_config import *

_cfg_file = NB_DIR / 'config.json'
if _cfg_file.exists():
    _cfg = json.loads(_cfg_file.read_text(encoding='utf-8'))
    for _k in ('DATA_SOURCE_MODE','FEATURE_SOURCE','TRAIN_START','TRAIN_END',
               'TEST_START','TEST_END','TARGET','RANDOM_STATE'):
        if _k in _cfg: globals()[_k] = _cfg[_k]

AUDIT_MODE = os.getenv("AUDIT_MODE", "0") == "1"
FEATURE_STEP3_CACHE = NB_DIR.parent / "cache" / "features_step3.parquet"
FEATURE_STEP3_CACHE.parent.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from keiba_ai.db_ultimate_loader import load_ultimate_training_frame
from keiba_ai.feature_engineering import add_derived_features
from keiba_ai.lightgbm_feature_optimizer import LightGBMFeatureOptimizer
from keiba_ai.constants import FUTURE_FIELDS, UNNECESSARY_COLUMNS

print(f"FEATURE_SOURCE = {FEATURE_SOURCE}")
print(f"TARGET         = {TARGET}")
print(f"AUDIT_MODE     = {AUDIT_MODE}")
print(f"STEP3_CACHE    = {FEATURE_STEP3_CACHE}")

FEATURE_SOURCE = existing
TARGET         = speed_deviation
AUDIT_MODE     = True
STEP3_CACHE    = C:\Users\yuki2\Documents\ws\keiba-ai-pro\cache\features_step3.parquet


In [2]:
# ── Step1: DB 読み込み ────────────────────────────────────────
_db = get_db_path(FEATURE_SOURCE)
print(f"DB: {_db}")

df_base = load_ultimate_training_frame(_db)
print(f"生データ: {len(df_base):,} 行 × {df_base.shape[1]} 列")

# 期間フィルタ（レース ID は YYYYMMDD... 形式）
df_train_raw = df_base[df_base['race_id'].str[:8] <= TRAIN_END].copy()
df_test_raw  = df_base[(df_base['race_id'].str[:8] >= TEST_START) &
                        (df_base['race_id'].str[:8] <= TEST_END)].copy()
print(f"学習期間 ({TRAIN_START}～{TRAIN_END}): {len(df_train_raw):,} 行")
print(f"テスト期間 ({TEST_START}～{TEST_END}):  {len(df_test_raw):,} 行")

DB: C:\Users\yuki2\Documents\ws\keiba-ai-pro\keiba\data\keiba_ultimate.db
→ Ultimate DBからデータ読み込み: C:\Users\yuki2\Documents\ws\keiba-ai-pro\keiba\data\keiba_ultimate.db
  絶対パス: C:\Users\yuki2\Documents\ws\keiba-ai-pro\keiba\data\keiba_ultimate.db
  存在チェック: True


  ✓ return_tables_ultimate: 50155レースの配当情報をロード


  ✓ holding_times_cache: 23969レース（バックフィル用）
  ✓ 575346行取得


  ✓ DataFrame変換: 575346行 × 123列


  ✓ num_horses: 欠損なし (575346件)
  ✓ venue コード解決: 573039件


  ✓ prev_race_class (ラグ): 510,812/575,346件 (88.8%)


  ✓ 型変換完了: 数値=72列, 文字列=3列, 合計=132列
  ✓ category: 57列
  ✓ distance: 575346件取得
  ✓ surface: 575346件取得
  ✓ メモリ最適化: 2641.24MB -> 338.98MB (2302.26MB, 87.17%)


生データ: 575,346 行 × 132 列


学習期間 (20130101～20170101): 383,632 行
テスト期間 (20170102～20180505):  103,839 行


In [3]:
# ── Step2: 追加テーブル読み込み ───────────────────────────────
# training_data  → last_training_time_3f / has_training_data / last_training_grade_encoded
# speed_figures  → sf_index_last / sf_index_2ago / sf_index_3ago / sf_index_trend
#
# ■ 前回の IN句方式の問題点と修正
#   原因: _target_race_ids が数千件 → SQLite の変数上限 (999) を超えて OperationalError
#   修正: race_id の先頭8桁が YYYYMMDD 形式であることを利用し、
#         日付範囲フィルタ (race_id BETWEEN ? AND ?) に変更
#   効果: 変数は最大4個 → 上限問題が永久に発生しない
#         idx_training_race_id がそのまま使われ高速
#
# speed_figures 列名: 実際の列名は index_last / index_2ago / index_3ago
#   (sf_index_1/2/3 は存在しない列名のため常に空 DataFrame になっていた)

import sqlite3, time as _time

# training_data: 実列名どおり
_TRAINING_COLS = ['race_id', 'horse_number', 'time_3f', 'grade', 'is_last_training']

# speed_figures: 実際の列名に修正
_SPEED_COLS = ['race_id', 'horse_number', 'index_last', 'index_2ago', 'index_3ago',
               'max_index', 'dist_max_index', 'course_max_index']

def _load_table_by_daterange(db_path, table: str, cols: list[str],
                              date_ranges: list[tuple[str, str]]) -> pd.DataFrame:
    """
    race_id の先頭8桁 (YYYYMMDD) を使った日付範囲フィルタでロード。
    IN句を使わないため SQLite の変数上限 (999) に引っかからない。
    date_ranges: [(start1, end1), (start2, end2), ...]  各要素は YYYYMMDD 文字列
    """
    if not db_path.exists():
        return pd.DataFrame()
    conn = sqlite3.connect(str(db_path))
    try:
        # 実在列だけ選択
        existing = {r[1] for r in conn.execute(f"PRAGMA table_info({table})").fetchall()}
        sel = [c for c in cols if c in existing]
        if not sel:
            return pd.DataFrame()

        # 日付範囲ごとに BETWEEN 句を組み立てる（変数は最大 2×len(date_ranges) 個）
        # race_id の先頭8桁で比較 → インデックスが有効
        where_clauses = " OR ".join(
            f"SUBSTR(race_id, 1, 8) BETWEEN ? AND ?" for _ in date_ranges
        )
        params = [v for (s, e) in date_ranges for v in (s, e)]
        sql = f"SELECT {','.join(sel)} FROM {table} WHERE {where_clauses}"
        return pd.read_sql(sql, conn, params=params)
    finally:
        conn.close()

_db_train = get_db_path(FEATURE_SOURCE)

# 学習期間 + テスト期間を date_ranges として渡す
_date_ranges = [(TRAIN_START, TRAIN_END), (TEST_START, TEST_END)]
print(f"対象期間: {_date_ranges}")

_t0 = _time.time()
training_df = _load_table_by_daterange(_db_train, 'training_data', _TRAINING_COLS, _date_ranges)
print(f"training_data:  {len(training_df):,} 行  ({_time.time()-_t0:.3f}s)")

_t1 = _time.time()
speed_figs_df = _load_table_by_daterange(_db_train, 'speed_figures', _SPEED_COLS, _date_ranges)
print(f"speed_figures:  {len(speed_figs_df):,} 行  ({_time.time()-_t1:.3f}s)")

if training_df.empty:
    print("⚠ training_data: 該当データなし → last_training_time_3f は NaN になります")
if speed_figs_df.empty:
    print("⚠ speed_figures: 該当データなし → sf_index_* は NaN になります")


対象期間: [('20130101', '20170101'), ('20170102', '20180505')]


training_data:  74,931 行  (0.229s)
speed_figures:  0 行  (0.008s)
⚠ speed_figures: 該当データなし → sf_index_* は NaN になります


In [4]:
# ── Step3: 特徴量生成（INV-01 パイプライン）──────────────────
# 本番コード: routers/train.py → keiba_ai.feature_engineering.add_derived_features
# 監査モードでは cache/features_step3.parquet を優先利用し、重処理を短縮する。

print("特徴量エンジニアリング開始...")
import time
from joblib import Parallel, delayed

def _build_features(_df_raw, _split_name: str):
    _t = time.time()
    _out = add_derived_features(
        _df_raw,
        full_history_df=df_base,
        training_df=training_df if len(training_df) > 0 else None,
        speed_figures_df=speed_figs_df if len(speed_figs_df) > 0 else None,
    ).copy()
    _out["_split"] = _split_name
    print(f"{_split_name} 特徴量生成完了: {len(_out):,} 行 × {_out.shape[1]} 列  ({time.time()-_t:.1f}s)")
    return _out

if AUDIT_MODE and FEATURE_STEP3_CACHE.exists():
    print(f"[AUDIT] cache hit: {FEATURE_STEP3_CACHE}")
    _df_step3 = pd.read_parquet(FEATURE_STEP3_CACHE)
else:
    _df_train_src = df_train_raw
    _df_test_src = df_test_raw

    if AUDIT_MODE:
        # 監査では分布を維持しつつ件数を削減する。
        _df_train_src = df_train_raw.sample(frac=0.35, random_state=RANDOM_STATE)
        _df_test_src = df_test_raw.sample(frac=0.35, random_state=RANDOM_STATE)
        print(f"[AUDIT] sample train={len(_df_train_src):,}, test={len(_df_test_src):,}")

    _t0 = time.time()
    _res = Parallel(n_jobs=2, prefer="threads")([
        delayed(_build_features)(_df_train_src, "train"),
        delayed(_build_features)(_df_test_src, "test"),
    ])
    _df_step3 = pd.concat(_res, ignore_index=True)
    print(f"Step3 total elapsed: {time.time()-_t0:.1f}s")

    if AUDIT_MODE:
        try:
            _df_step3.to_parquet(FEATURE_STEP3_CACHE, index=False)
            print(f"[AUDIT] cache saved: {FEATURE_STEP3_CACHE}")
        except Exception as _e:
            print(f"[AUDIT] cache save skipped: {_e}")

df_train = _df_step3[_df_step3["_split"] == "train"].drop(columns=["_split"]).reset_index(drop=True)
df_test = _df_step3[_df_step3["_split"] == "test"].drop(columns=["_split"]).reset_index(drop=True)

print(f"学習 特徴量生成完了: {len(df_train):,} 行 × {df_train.shape[1]} 列")
print(f"テスト 特徴量生成完了: {len(df_test):,} 行 × {df_test.shape[1]} 列")

# apply 系の残存箇所を可視化（除去対象の監査用）
_fe_file = NB_DIR.parent / "keiba" / "keiba_ai" / "feature_engineering.py"
_patterns = ["apply(", "apply(axis=1", "groupby.apply", "rolling.apply"]
if _fe_file.exists():
    _txt = _fe_file.read_text(encoding="utf-8", errors="ignore")
    print("\n[apply usage scan]")
    for _p in _patterns:
        print(f"  {_p:<14}: {_txt.count(_p)}")

特徴量エンジニアリング開始...
[AUDIT] sample train=134,271, test=36,344


test 特徴量生成完了: 36,344 行 × 385 列  (67.2s)


train 特徴量生成完了: 134,271 行 × 385 列  (68.0s)
Step3 total elapsed: 68.2s
[AUDIT] cache save skipped: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.


学習 特徴量生成完了: 134,271 行 × 384 列
テスト 特徴量生成完了: 36,344 行 × 384 列

[apply usage scan]
  apply(        : 16
  apply(axis=1  : 0
  groupby.apply : 1
  rolling.apply : 0


In [5]:
# ── Step4: 目的変数生成 → FUTURE_FIELDS 除外（INV-01 順序）────
# ★ ターゲット抽出を leakage 除外より先に行う
#
# 【speed_deviation について】
#   DB に "speed_deviation" 列は存在しない。
#   本番コード (keiba_ai/train.py _make_target) と同様に
#   time_seconds / distance から on-the-fly で計算する。
#   df.get("speed_deviation") では列が無く常に空 Series になるため NG。

def _make_target(df: pd.DataFrame, target: str) -> pd.Series:
    """keiba/keiba_ai/train.py の _make_target と同ロジック"""
    _finish = pd.to_numeric(
        df.get('finish', pd.Series(dtype=float)), errors='coerce'
    )

    if target == 'win':
        return (_finish == 1).astype(int)

    elif target == 'place3':
        return (_finish <= 3).fillna(False).astype(int)

    elif target == 'speed_deviation':
        # ★ 列名検索ではなく time_seconds / distance から計算（本番と同ロジック）
        _ts   = pd.to_numeric(df.get('time_seconds',  pd.Series(dtype=float)), errors='coerce')
        _dist = pd.to_numeric(df.get('distance',      pd.Series(dtype=float)), errors='coerce')

        if _ts.isna().all():
            raise ValueError(
                "[TARGET ERROR] time_seconds 列が全 NaN です。\n"
                "  ▸ DB に time_seconds が格納されているか確認してください。\n"
                f"  ▸ df の列: {df.columns.tolist()[:20]} ..."
            )
        if _dist.isna().all():
            raise ValueError(
                "[TARGET ERROR] distance 列が全 NaN です。"
            )

        _spd = _dist / _ts.replace(0, np.nan)

        # レース内 z-score（race_id でグループ化）
        if 'race_id' in df.columns:
            _grp_mean = _spd.groupby(df['race_id']).transform('mean')
            _grp_std  = _spd.groupby(df['race_id']).transform('std').replace(0, np.nan)
        else:
            _grp_mean = _spd.mean()
            _grp_std  = _spd.std() or np.nan

        _result = (_spd - _grp_mean) / _grp_std
        print(f"  [speed_deviation] 計算元: time_seconds={_ts.notna().sum()}件有効 / "
              f"distance={_dist.notna().sum()}件有効 / "
              f"speed_deviation={_result.notna().sum()}件有効")
        return _result

    raise ValueError(f"Unknown target: {target}")

# ── ターゲット先取り（leakage 除外前）─────────────────────────
y_train = _make_target(df_train, TARGET)
y_test  = _make_target(df_test,  TARGET)

if TARGET == 'speed_deviation':
    print(f"y_train: mean={y_train.mean():.3f}  std={y_train.std():.3f}  "
          f"NaN={y_train.isna().sum()}  len={len(y_train)}")
    print(f"y_test:  mean={y_test.mean():.3f}  std={y_test.std():.3f}  "
          f"NaN={y_test.isna().sum()}  len={len(y_test)}")
else:
    print(f"y_train: {y_train.sum()} 正例 / {len(y_train)} 行 ({y_train.mean()*100:.1f}%)")
    print(f"y_test:  {y_test.sum()} 正例 / {len(y_test)} 行 ({y_test.mean()*100:.1f}%)")

assert len(y_train) > 0, \
    f"y_train が空です。TARGET='{TARGET}'"
assert len(y_train) == len(df_train), \
    f"サイズ不一致: y_train={len(y_train)}, df_train={len(df_train)}"

# ── FUTURE_FIELDS / UNNECESSARY_COLUMNS 除外（INV-01）──────────
def _drop_leakage(df: pd.DataFrame) -> pd.DataFrame:
    _drop = list(FUTURE_FIELDS) + list(UNNECESSARY_COLUMNS)
    _present = [c for c in _drop if c in df.columns]
    if _present:
        print(f"  除外: {len(_present)} 列")
        df = df.drop(columns=_present, errors='ignore')
    return df

print("\n学習データ:")
df_train = _drop_leakage(df_train)
print("テストデータ:")
df_test  = _drop_leakage(df_test)
print(f"学習: {df_train.shape}  テスト: {df_test.shape}")


  [speed_deviation] 計算元: time_seconds=132505件有効 / distance=134271件有効 / speed_deviation=129815件有効
  [speed_deviation] 計算元: time_seconds=35967件有効 / distance=36344件有効 / speed_deviation=35375件有効
y_train: mean=0.000  std=0.875  NaN=4456  len=134271
y_test:  mean=-0.000  std=0.878  NaN=969  len=36344

学習データ:
  除外: 179 列
テストデータ:
  除外: 179 列
学習: (134271, 226)  テスト: (36344, 226)


In [6]:
# ── Step5: 目的変数確認（生成は Step4 で実施済み）─────────────
# ★ y_train / y_test は Step4 で leakage 除外前に生成済み
#   ここでは再生成せず、結果を確認するだけ

print(f"TARGET = '{TARGET}'")
print(f"y_train: len={len(y_train)}  "
      + (f"NaN={y_train.isna().sum()}  mean={y_train.mean():.4f}  std={y_train.std():.4f}"
         if TARGET == 'speed_deviation' else
         f"{y_train.sum()} 正例 ({y_train.mean()*100:.1f}%)"))
print(f"y_test : len={len(y_test)}  "
      + (f"NaN={y_test.isna().sum()}  mean={y_test.mean():.4f}  std={y_test.std():.4f}"
         if TARGET == 'speed_deviation' else
         f"{y_test.sum()} 正例 ({y_test.mean()*100:.1f}%)"))

assert len(y_train) > 0, f"y_train が空です (TARGET='{TARGET}')"
print("✓ y_train / y_test OK")


TARGET = 'speed_deviation'
y_train: len=134271  NaN=4456  mean=0.0000  std=0.8755
y_test : len=36344  NaN=969  mean=-0.0000  std=0.8780
✓ y_train / y_test OK


In [7]:
# ── Step6: LightGBMFeatureOptimizer 変換 ──────────────────────
# Bug #5 修正: テストデータは optimizer.transform() を使用する。
# fit_transform を使うと Encoder がテストデータの分布で再学習され、
# 学習時とカテゴリマッピングが乖離する。

optimizer = LightGBMFeatureOptimizer()

X_train, cat_features = optimizer.fit_transform(df_train.copy(), target_col='finish')

# テストは transform のみ（学習時の Encoder を再利用）
X_test = optimizer.transform(df_test.copy())

# ID・ターゲット列を除外
_id_cols = ['race_id','horse_id','jockey_id','trainer_id','owner_id','finish','finish_position']
X_train = X_train.drop(columns=[c for c in _id_cols if c in X_train.columns], errors='ignore')
X_test  = X_test.drop(columns=[c for c in _id_cols if c in X_test.columns],  errors='ignore')

# テストを学習と列一致させる（欠落列=0埋め、余分列=除去）
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"カテゴリ特徴量: {len(cat_features)}")

# object 型残存確認
_obj_tr = X_train.select_dtypes(include='object').columns.tolist()
_obj_te = X_test.select_dtypes(include='object').columns.tolist()
if _obj_tr or _obj_te:
    print(f"⚠ object 型残存 - train: {_obj_tr}  test: {_obj_te}")
else:
    print("✓ object 型なし（LightGBM 投入可能）")



【LightGBM特徴量最適化】学習モード

【1. 低カーディナリティ カテゴリカル変数】
処理: Label Encoding → LightGBMのcategorical_feature指定


  ✓ venue                          → venue_encoded                  (25種類)
  ✓ venue_code                     → venue_code_encoded             (31種類)
  ✓ track_type                     → track_type_encoded             (4種類)


  ✓ surface                        → surface_encoded                (4種類)
  ✓ weather                        → weather_encoded                (7種類)
  ✓ field_condition                → field_condition_encoded        (5種類)
  ✓ race_class                     → race_class_encoded             (24種類)
  ✓ course_direction               → course_direction_encoded       (4種類)
  ✓ corner_radius                  → corner_radius_encoded          (4種類)

  元のカテゴリカルカラムを削除: ['venue', 'venue_code', 'track_type', 'surface', 'weather', 'field_condition', 'race_class', 'course_direction', 'corner_radius']



【2. 高カーディナリティ文字列列】
処理: 削除（統計特徴量は feature_engineering.py で計算済み）
  ✓ horse_name → 削除
  ✓ sire → 削除


  ✓ dam → 削除
  ✓ damsire → 削除

【3. 数値変数】
処理: そのまま使用（LightGBMは自動でスケーリング不要）
  ✓ 数値特徴量: 70個
    - horse_number
    - horse_weight
    - horse_weight_change
    - distance
    - num_horses
    - race_num
    - kai
    - day
    - days_since_last_race
    - distance_change
    ... 他60個

【4. バイナリ特徴量】
処理: そのまま使用（0/1エンコード済み）
  ✓ バイナリ特徴量: 0個

【5. リスト型変数】
処理: 統計値（平均、分散など）に変換

【6. ダミー変数】
処理: そのまま使用（既にバイナリ化済み）
  ✓ 性別ダミー: ['sex_code']

【7. ID系変数】
処理: そのまま保持（統計計算に使用、学習時は除外推奨）
  ✓ ID特徴量: ['race_id', 'horse_id', 'jockey_id', 'trainer_id']
    → 学習時にはこれらを除外してください

【8. 日時変数】
処理: 数値化（年/月/日/曜日など）
  ✓ date → date_year, date_month, date_day, date_dayofweek

【9. 不要な変数（削除推奨）】
  ✓ is_first_race → 削除


  ✓ surface_encoded → 削除
  ✓ track_type_encoded → 削除


  ✓ course_direction_encoded → 削除
  ✓ corner_radius_encoded → 削除
  ✓ 性別ダミー（重複）削除: ['sex_code']



  ⚠️  欠損率90%超の列を除去 (39列): ['prev2_race_class', 'prev3_race_class', 'prev4_race_class', 'prev5_race_class', 'running_style_num', 'jockey_changed', 'first_jockey', 'lap_200m']...


  ⚠️  [fix-D] ゼロ分散列を除去 (32列): ['sf_index_2ago', 'sf_index_3ago', 'sf_max_index', 'sf_course_max_index', 'sf_dist_max_index', 'sf_index_trend', 'training_comment_score', 'horse_speed_rank_pct_is_missing', 'class_drop_is_missing', 'sire_surface_win_rate_is_missing', 'sire_dist_band_win_rate_is_missing', 'damsire_surface_win_rate_is_missing', 'damsire_dist_band_win_rate_is_missing', 'horse_field_win_rate_is_missing', 'horse_field_races_is_missing', 'horse_weather_win_rate_is_missing', 'horse_weather_races_is_missing', 'jockey_field_win_rate_is_missing', 'jockey_field_races_is_missing', 'sire_field_win_rate_is_missing', 'sire_field_races_is_missing', 'damsire_field_win_rate_is_missing', 'damsire_field_races_is_missing', 'kai_num_is_missing', 'day_num_is_missing', 'training_comment_score_is_missing', 'sf_index_2ago_is_missing', 'sf_index_3ago_is_missing', 'sf_max_index_is_missing', 'sf_course_max_index_is_missing', 'sf_dist_max_index_is_missing', 'sf_index_trend_is_missing']

【最適化完了】
  元のカラ

  ✓ 変換完了: 226カラム
X_train: (134271, 146)
X_test:  (36344, 146)
カテゴリ特徴量: 9
✓ object 型なし（LightGBM 投入可能）


In [8]:
# ── Step7: feature_summary.csv 出力 ──────────────────────────
_feat_rows = []
for col in X_train.columns:
    s = X_train[col]
    _feat_rows.append({
        'feature': col,
        'dtype': str(s.dtype),
        'null_rate': round(s.isnull().mean()*100, 2),
        'mean': round(s.mean(), 4) if pd.api.types.is_numeric_dtype(s) else None,
        'std':  round(s.std(),  4) if pd.api.types.is_numeric_dtype(s) else None,
        'is_categorical': col in cat_features,
    })
feature_summary = pd.DataFrame(_feat_rows)
_out = REPORTS_DIR / 'feature_summary.csv'
feature_summary.to_csv(_out, index=False, encoding='utf-8-sig')
print(f"feature_summary.csv 保存: {_out} ({len(feature_summary)} 列)")
display(feature_summary.head(10))

feature_summary.csv 保存: C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\reports\feature_summary.csv (146 列)


,feature,dtype,null_rate,mean,std,is_categorical
0,horse_number,int8,0.00,6.678900,4.044200,False
1,prev_race_distance,float32,0.00,1262.993652,674.393616,False
2,prev2_race_finish,float32,25.42,6.530800,3.980100,False
3,prev2_race_distance,float32,0.00,1156.533936,729.636902,False
4,distance,int16,0.00,1399.343000,537.812200,False
5,num_horses,int8,0.00,12.370300,3.322700,False
6,kai,int8,0.00,6.445900,6.129200,False
7,day,int8,0.00,3.868200,2.403600,False
8,prev_race_finish,float32,18.83,6.660400,4.040500,False
9,prev3_race_finish,float32,31.93,6.422600,3.942800,False


In [9]:
# ── 中間データ保存（features.pkl）────────────────────────────
import joblib

# ── 保存前バリデーション ───────────────────────────────────────
assert len(X_train) > 0,  f"X_train が空です"
assert len(y_train) > 0,  f"y_train が空です (TARGET='{TARGET}')"
assert len(X_test)  > 0,  f"X_test が空です"
assert len(y_test)  > 0,  f"y_test が空です"
assert len(X_train) == len(y_train), \
    f"サイズ不一致: X_train={len(X_train)}, y_train={len(y_train)}"
assert len(X_test) == len(y_test), \
    f"サイズ不一致: X_test={len(X_test)}, y_test={len(y_test)}"
print(f"✓ 保存前チェック OK: X_train={X_train.shape}  y_train={len(y_train)}")

_feat_cache = NB_DIR / 'data' / 'features'
_feat_cache.mkdir(parents=True, exist_ok=True)

joblib.dump({'X_train': X_train, 'X_test': X_test,
             'y_train': y_train, 'y_test': y_test,
             'df_train': df_train, 'df_test': df_test,
             'cat_features': cat_features, 'optimizer': optimizer},
            _feat_cache / 'features.pkl')
print(f"特徴量キャッシュ保存: {_feat_cache / 'features.pkl'}")


✓ 保存前チェック OK: X_train=(134271, 146)  y_train=134271


特徴量キャッシュ保存: C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\data\features\features.pkl
